In [1]:
import numpy as np
import scipy.optimize 

#### Ottimizzazione Numerica e L-BFGS

L'ottimizzazione numerica gioca un ruolo fondamentale nell'addestramento dei modelli di classificazione, come ad esempio la Regressione Logistica. A differenza dei modelli gaussiani, per i quali sono disponibili espressioni in forma chiusa per ottenere le soluzioni di Maximum Likelihood, la Regressione Logistica non ammette una soluzione analitica diretta. Il modello si ottiene infatti minimizzando la cross-entropia media calcolata tra le predizioni del modello stesso e le etichette effettivamente osservate, il che corrisponde alla ricerca della soluzione di Maximum Likelihood. Di conseguenza, per trovare il punto di minimo di questa funzione obiettivo, o in modo del tutto equivalente il massimizzatore delle verosimiglianze di classe, risulta strettamente necessario ricorrere all'utilizzo di algoritmi di ottimizzazione numerica.

#### Gradient Descent

Gli algoritmi di ottimizzazione numerica cercano i minimi di una data funzione $f(x)$ rispetto al suo argomento. Un metodo iterativo tra i più semplici per individuare un minimo locale della funzione $f$ è la discesa del gradiente, o gradient descent. Partendo da un punto iniziale $x_{t}$, questo algoritmo analizza l'andamento della funzione al fine di cercare una direzione di discesa. Tale direzione è data matematicamente dall'opposto del gradiente della funzione valutato in quel punto. A ogni singola iterazione, l'algoritmo calcola il punto successivo muovendosi lungo la direzione di discesa partendo da $x_{t}$ e compiendo un passo di dimensione $\alpha_{t}$. Questo passaggio cruciale è formalizzato dall'equazione di aggiornamento iterativo:

$$x_{t+1}=x_{t}-\alpha_{t}\nabla_{x}f(x)$$

Sotto alcune ipotesi non stringenti relative alla dimensione del passo $\alpha_{t}$, come ad esempio le condizioni per cui $\alpha_{t}\rightarrow0$ e simultaneamente $\sum_{t=1}^{\infty}\alpha_{t}\rightarrow\infty$, l'algoritmo garantisce la convergenza verso un minimo locale della funzione $f$. Uno degli svantaggi principali della discesa del gradiente risiede tuttavia nella sua lentezza. Per riuscire ad accelerare la convergenza verso il minimo, è possibile prendere in considerazione le informazioni del secondo ordine del dominio della funzione, introducendo il calcolo dell'Hessiana.

#### L-BFGS

A questo preciso scopo si inserisce l'algoritmo L-BFGS, un metodo avanzato che si basa sulla costruzione incrementale di un'approssimazione della matrice Hessiana, utilizzata per identificare una direzione di ricerca ottimale $p_{t}$ durante ogni iterazione. Una volta stabilita questa direzione di ricerca, l'algoritmo procede determinando una dimensione accettabile per il passo $\alpha_{t}$ per quella specifica direzione, impiegando poi congiuntamente direzione e dimensione del passo per aggiornare la soluzione corrente. L'implementazione pratica di questo ottimizzatore è disponibile nell'ecosistema Python tramite la libreria SciPy, interfacciandosi con il solutore numerico attraverso la funzione `scipy.optimize.fmin_l_bfgs_b`. Questa procedura richiede obbligatoriamente almeno due argomenti principali per essere eseguita: `func`, che rappresenta l'effettiva funzione obiettivo che si intende minimizzare, e `x0`, che definisce il valore o array di partenza per l'inizializzazione dell'algoritmo.

#### Calcolo del Gradiente

Per operare in maniera corretta, l'algoritmo L-BFGS richiede sia il calcolo della funzione obiettivo sia la conoscenza del suo gradiente. Lo strumento offre diverse alternative concettuali per passare il calcolo del gradiente all'interno della computazione. La prima modalità consiste nel costruire la funzione `func` affinché restituisca direttamente una tupla strutturata esattamente come $(f(x),\nabla_{x}f(x))$. Un approccio alternativo sfrutta il parametro opzionale `fprime`, mediante il quale si fornisce al sistema una seconda funzione interamente dedicata al calcolo matematico del gradiente; in questo scenario specifico, la funzione principale `func` dovrà limitarsi a restituire esclusivamente il valore obiettivo $f(x)$.

L'ultima opzione implementabile consiste nell'affidare completamente all'implementazione sottostante il calcolo di un gradiente approssimato, impostando l'argomento `approx_grad = True` durante la chiamata. Anche in quest'ultimo caso, la funzione `func` elaborerà unicamente il valore obiettivo $f(x)$. Questa comoda modalità operativa non richiede allo sviluppatore lo sforzo manuale di derivare formalmente la funzione e tradurla in codice, poiché l'approssimazione del gradiente viene calcolata in totale autonomia tramite il metodo numerico delle differenze finite. Ad esempio, il sistema valuta numericamente il gradiente calcolando la seguente equazione per le varie coordinate:

In [2]:
dataset = np.loadtxt('../trainData.txt', delimiter=',')
D=dataset[:,:6].T
L=dataset[:,6]

labels={
    1:"True",
    0:"False"
}

### Preparazione del dataset
In questa cella carichiamo `trainData.txt` e separiamo:
- la matrice delle feature $D \in \mathbb{R}^{6 \times N}$
- il vettore delle etichette $L \in \{0,1\}^N$

L'orientamento scelto (feature sulle righe, campioni sulle colonne) è coerente con tutte le funzioni di stima successive.

In [ ]:
def getMu(D):
    return D.sum(axis=1)/D.shape[1]

def vRow(vet):
    return vet.reshape((1, vet.size))
def vCol(vet):
     return vet.reshape((vet.size, 1))

def getC(D):
    mu=getMu(D)
    mu=vCol(mu)
    Dc=D-mu
    return (Dc @ Dc.T)/Dc.shape[1]

def compute_Sb_Sw(D, L):
    Sb = 0
    Sw = 0
    muGlobal = vCol(D.mean(1))
    for i in np.unique(L):
        DCls = D[:, L == i]
        mu = vCol(DCls.mean(1))
        Sb += (mu - muGlobal) @ (mu - muGlobal).T * DCls.shape[1]
        Sw += (DCls - mu) @ (DCls - mu).T
    return Sb / D.shape[1], Sw / D.shape[1]
    
def split_db_2to1(D, L, seed=0):
    nTrain = int(D.shape[1]*2.0/3.0)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    idxTrain = idx[0:nTrain]
    idxTest = idx[nTrain:]
    DTR = D[:, idxTrain]
    DVAL = D[:, idxTest]
    LTR = L[idxTrain]
    LVAL = L[idxTest]
    return (DTR, LTR), (DVAL, LVAL)

### Funzioni statistiche di base
Qui definiamo utilità necessarie per tutta la pipeline:
- media e covarianza campionaria (`getMu`, `getC`)
- reshape a vettore riga/colonna (`vRow`, `vCol`)
- matrici di dispersione intra/inter-classe (`compute_Sb_Sw`)
- split train/validation 2:1 (`split_db_2to1`)

Queste funzioni rendono il notebook modulare e riusabile anche per altri modelli generativi/discriminativi.

In [4]:
(DTR, LTR), (DVAL, LVAL)=split_db_2to1(D,L)

### Split training/validation
Da questo punto in poi stimiamo i parametri solo su `DTR, LTR` e misuriamo le prestazioni su `DVAL, LVAL` per evitare valutazioni ottimistiche.

La logica è:
1. training del modello su train
2. calcolo score su validation
3. decisione Bayesiana e metriche (error rate, minDCF, actualDCF).

In [5]:
def get_confusion_mat(predictions,L):
    labels=np.unique(L)
    M=np.zeros((len(labels),len(labels)))

    predictions_int = np.array(predictions).astype(int)
    L_int = np.array(L).astype(int)

    for i in range(len(predictions_int)):
        M[predictions_int[i]][L_int[i]]+=1
    
    return M
def DCF_u(predictions,L,prior,Cfn,Cfp):
    C=np.array([[0,Cfn],[Cfp,0]])
    M=get_confusion_mat(predictions,L)
    cols_sum=np.sum(M,axis=0)
    R=M/cols_sum 
    expected_cost_per_class=np.sum(R*C,axis=0)
    prior_array = np.array([1 - prior, prior])
    DCFu=np.sum(expected_cost_per_class * prior_array)
    return DCFu

def DCF_norm(predictions,L,prior,Cfn,Cfp):
    B_dummy=min(prior*Cfn,(1-prior)*Cfp)
    dcf_u=DCF_u(predictions,L,prior,Cfn,Cfp)
    return dcf_u/B_dummy
def get_bayes_decision(llr,pi,Cfn,Cfp):
    pi_Ht=pi
    pi_Hf=1-pi
    t = -np.log((pi_Ht * Cfn) / (pi_Hf * Cfp))
    predictions=np.where(llr>t,1,0)

    return predictions

def compute_min_dcf(llr, L, pi, Cfn, Cfp):
    # Ordinamento degli score (LLR) per ricavare s_1...s_M
    sorted_llr = np.sort(llr)
    
    # Creazione del vettore delle soglie includendo gli estremi -inf e +inf
    thresholds = np.concatenate([np.array([-np.inf]), sorted_llr, np.array([np.inf])])
    
    # Inizializzazione del minimo a un valore infinitamente grande
    min_dcf = np.inf
    
    # Iterazione su tutte le possibili soglie t
    for t in thresholds:
        # Calcolo delle predizioni basate sulla soglia corrente
        predictions = np.where(llr > t, 1, 0)

        # Calcolo della DCF normalizzata
        dcf_n = DCF_norm(predictions,L,pi,Cfn,Cfp)
        
        # Aggiornamento del valore minimo se la soglia corrente risulta più performante
        if dcf_n < min_dcf:
            min_dcf = dcf_n
            
    return min_dcf

### Metriche Bayesiane: confusion matrix e DCF
Queste funzioni implementano la valutazione decisionale in ottica Bayes.

- `DCF_u`: costo non normalizzato
- `DCF_norm`: costo normalizzato rispetto al sistema dummy
- `compute_min_dcf`: cerca la soglia ottima scorrendo tutti i possibili threshold sugli score

Ricorda che:
$$\mathrm{minDCF} \leq \mathrm{actualDCF}$$
e il gap tra i due valori riflette il livello di calibrazione degli score.

In [6]:
def trainLogReg(DTR,LTR,l):
    ZTR=2*LTR-1
    n=DTR.shape[1]

    def logreg_obj(v):
        w,b=v[0:-1],v[-1]
        w=w.reshape(-1,1)
        S=(np.dot(w.T,DTR)+b).ravel()
        loss_terms=np.logaddexp(0,-ZTR*S)
        J=(l/2)*np.linalg.norm(w)**2+np.mean(loss_terms)
        G=-ZTR/(1.0 + np.exp(ZTR*S))
        #gradienti
        grad_w=l*w.ravel()+np.mean(G*DTR,axis=1)
        grad_b=np.mean(G)
        v_grad=np.hstack([grad_w,grad_b])
        return J,v_grad
    
    x0 = np.zeros(DTR.shape[0]+1) #??perchè
    xf,f_min,d=scipy.optimize.fmin_l_bfgs_b(logreg_obj,x0=x0,approx_grad=False)

    return xf,f_min

### Addestramento della Logistic Regression lineare
Qui definiamo la funzione obiettivo regolarizzata:
$$J(w,b)=\frac{\lambda}{2}\|w\|^2 + \frac{1}{N}\sum_{i=1}^{N}\log\left(1+e^{-z_i(w^Tx_i+b)}\right)$$
con $z_i\in\{-1,+1\}$.

Il solver L-BFGS minimizza $J$ usando anche il gradiente analitico, migliorando velocità e stabilità rispetto al gradiente numerico approssimato.

In [7]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 2.377797e-01
Error Rate: 9.3%
DCF min: 0.36397529441884274
Actual DCF:0.9440924219150024

Lambda: 0.00031622776601683794
J ottima: 2.391006e-01
Error Rate: 9.3%
DCF min: 0.3649673579109063
Actual DCF:0.9531650025601638

Lambda: 0.001
J ottima: 2.430980e-01
Error Rate: 9.3%
DCF min: 0.3649673579109063
Actual DCF:0.9531650025601638

Lambda: 0.0031622776601683794
J ottima: 2.543260e-01
Error Rate: 9.2%
DCF min: 0.36411930363543266
Actual DCF:0.9350198412698413

Lambda: 0.01
J ottima: 2.815594e-01
Error Rate: 9.2%
DCF min: 0.3611431131592422
Actual DCF:0.9350198412698413

Lambda: 0.03162277660168379
J ottima: 3.353240e-01
Error Rate: 9.2%
DCF min: 0.3621351766513057
Actual DCF:0.9340277777777778

Lambda: 0.1
J ottima: 4.198415e-01
Error Rate: 9.2%
DCF min: 0.36411930363543266
Actual DCF:0.9431003584229389

Lambda: 0.31622776601683794
J ottima: 5.223593e-01
Error Rate: 9.3%
DCF min: 0.36397529441884274
Actual DCF:0.9521729390681003

Lambda: 1.0
J ottima: 6.107602e-0

### Esperimento 1: modello lineare su training completo
In questa fase testiamo diversi valori di $\lambda$ su tutto il training set.

Interpretazione delle metriche:
- **Error Rate**: accuratezza di classificazione con soglia fissa
- **minDCF**: migliore costo possibile variando la soglia
- **actualDCF**: costo al punto operativo scelto (con prior $\pi_T$ e costi)

Se `actualDCF` è molto maggiore di `minDCF`, il modello discrimina ma gli score non sono ben calibrati per quel punto operativo.

In [11]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR[:,::50],LTR[::50],l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 1.197749e-01
Error Rate: 11.1%
DCF min: 0.4466045826932924
Actual DCF:1.5050403225806452

Lambda: 0.00031622776601683794
J ottima: 1.255871e-01
Error Rate: 10.9%
DCF min: 0.4446204557091654
Actual DCF:1.4778225806451613

Lambda: 0.001
J ottima: 1.386344e-01
Error Rate: 10.8%
DCF min: 0.4487327188940092
Actual DCF:1.4435163850486432

Lambda: 0.0031622776601683794
J ottima: 1.636818e-01
Error Rate: 10.5%
DCF min: 0.44972478238607266
Actual DCF:1.3729198668714797

Lambda: 0.01
J ottima: 2.070173e-01
Error Rate: 10.4%
DCF min: 0.44065220174091135
Actual DCF:1.3466941884280594

Lambda: 0.03162277660168379
J ottima: 2.759238e-01
Error Rate: 9.8%
DCF min: 0.4147145417306707
Actual DCF:1.2216621863799282

Lambda: 0.1
J ottima: 3.738010e-01
Error Rate: 9.8%
DCF min: 0.39884152585765487
Actual DCF:1.1388728878648233

Lambda: 0.31622776601683794
J ottima: 4.895101e-01
Error Rate: 9.2%
DCF min: 0.3887768817204301
Actual DCF:0.8865367383512546

Lambda: 1.0
J ottima: 5.90643

### Esperimento 2: training ridotto (1 campione ogni 50)
Qui ripetiamo la stessa procedura con pochissimi dati di training per osservare l'effetto del regime a bassa numerosità.

Aspettativa teorica:
- aumento della varianza di stima dei parametri
- maggiore sensibilità alla regolarizzazione
- possibile peggioramento di stabilità e costo decisionale su validation.

In [12]:
def weighted_trainLogReg(DTR,LTR,l,piT):
    ZTR=2*LTR-1 #definizione di z
    n=DTR.shape[1] #numero di campioni

    def weighed_logreg_obj(v):
        w,b=v[0:-1],v[-1]
        w=w.reshape(-1,1)

        S=(np.dot(w.T,DTR)+b).ravel()
        nT=(LTR==1).sum()
        nF=(LTR==0).sum()
        xi=np.where(ZTR==1,piT/nT,(1-piT)/nF)
        weighed_loss_terms=xi*np.logaddexp(0,-ZTR*S)
        J=(l/2)*np.linalg.norm(w)**2+np.sum(weighed_loss_terms)
        G=-ZTR/(1.0 + np.exp(ZTR*S))
        #gradienti
        grad_w=l*w.ravel()+np.sum(xi*G*DTR,axis=1)
        grad_b=np.sum(xi*G)
        v_grad=np.hstack([grad_w,grad_b])

        return J, v_grad

    x0 = np.zeros(DTR.shape[0] + 1)
    xf, f_min, d = scipy.optimize.fmin_l_bfgs_b(func=weighed_logreg_obj, x0=x0, approx_grad=False)
    
    return xf, f_min


### Logistic Regression pesata (prior-weighted)
Questa variante introduce pesi diversi per campioni positivi/negativi in funzione del prior target $\pi_T$.

L'obiettivo è allineare il training al punto operativo applicativo, penalizzando maggiormente gli errori sulla classe più critica.

Nota: anche con training pesato, può restare necessario calibrare i punteggi per ridurre il gap tra `actualDCF` e `minDCF`.

In [13]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=weighted_trainLogReg(DTR,LTR,l,pi_T)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    
    S_val=np.dot(w_opt.T,DVAL)+b_opt
    LP=(S_val>0).astype(int)
    error_rate = np.mean(LP != LVAL)
    llr=S_val-target_prior_log_odds
    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)

    predictions_bayes = get_bayes_decision(llr, pi_T, 1, 1)
    act_dcf = DCF_norm(predictions_bayes, LVAL, pi_T, 1, 1)

    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")

Lambda: 0.0001
J ottima: 1.295710e-01
Error Rate: 16.9%
DCF min: 0.37205581157194056
Actual DCF:0.407050051203277

Lambda: 0.00031622776601683794
J ottima: 1.307709e-01
Error Rate: 17.0%
DCF min: 0.3700716845878136
Actual DCF:0.4009536610343062

Lambda: 0.001
J ottima: 1.343060e-01
Error Rate: 17.5%
DCF min: 0.36992767537122373
Actual DCF:0.4128584229390681

Lambda: 0.0031622776601683794
J ottima: 1.436651e-01
Error Rate: 18.6%
DCF min: 0.36695148489503326
Actual DCF:0.43269969278033793

Lambda: 0.01
J ottima: 1.642912e-01
Error Rate: 21.8%
DCF min: 0.36298323092677925
Actual DCF:0.4487007168458782

Lambda: 0.03162277660168379
J ottima: 2.002131e-01
Error Rate: 29.6%
DCF min: 0.36397529441884274
Actual DCF:0.5963741679467486

Lambda: 0.1
J ottima: 2.473025e-01
Error Rate: 46.1%
DCF min: 0.36482334869431643
Actual DCF:0.9146825396825398

Lambda: 0.31622776601683794
J ottima: 2.889006e-01
Error Rate: 50.4%
DCF min: 0.36397529441884274
Actual DCF:1.0

Lambda: 1.0
J ottima: 3.117064e-01
Er

### Feature expansion quadratica
Per aumentare la capacità espressiva del modello lineare, espandiamo ogni campione con termini quadratici $xx^T$ oltre ai termini lineari.

In questo modo la decisione resta lineare nello spazio espanso ma diventa non lineare nello spazio originale.

Rischio/beneficio atteso: migliore separazione di pattern complessi, ma maggiore rischio di overfitting senza regolarizzazione adeguata.

In [9]:
def expand_features(D):
    n_features = D.shape[0]
    n_samples = D.shape[1]
    
    # La nuova dimensionalità sarà data dal quadrato delle feature originali (xx^T)
    # sommato al numero delle feature originali (x)
    expanded_D = np.zeros((n_features**2 + n_features, n_samples))
    
    for i in range(n_samples):
        # Estrazione del singolo campione come vettore colonna
        x = D[:, i:i+1]
        
        # Calcolo della matrice dei prodotti incrociati
        xxT = np.dot(x, x.T)
        
        # Vettorizzazione della matrice e concatenazione con il termine lineare
        phi_x = np.vstack([xxT.reshape(-1, 1), x])
        
        # Inserimento del campione espanso nella nuova matrice
        expanded_D[:, i:i+1] = phi_x
        
    return expanded_D

# Esempio di utilizzo prima di addestrare il modello:
DTR_quad = expand_features(DTR)
DVAL_quad = expand_features(DVAL)

### Esperimento 3: Logistic Regression quadratica
Infine valutiamo il modello dopo espansione delle feature, confrontando le metriche con la versione lineare.

Indicazioni di lettura dei risultati:
- se cala `minDCF`, la separabilità intrinseca è migliorata
- se cala anche `actualDCF`, oltre a separare meglio gli score sono anche più adatti al punto operativo
- se `actualDCF` non migliora quanto `minDCF`, conviene considerare una fase di calibrazione finale.

In [15]:
lambdas=np.logspace(-4,2,13)
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))

for l in lambdas:
    v_opt,J_min=trainLogReg(DTR_quad,LTR,l)
    w_opt=v_opt[:-1]
    b_opt=v_opt[-1]
    S_val=np.dot(w_opt.T,DVAL_quad)+b_opt
    LP=(S_val>0).astype(int)
    error_rate=np.mean(LP!=LVAL)
    llr=S_val-target_prior_log_odds

    min_dcf=compute_min_dcf(llr,LVAL,pi_T,1,1)
    predictions_bayes=get_bayes_decision(llr,pi_T,1,1)
    act_dcf=DCF_norm(predictions_bayes,LVAL,pi_T,1,1)
    print(f"Lambda: {l}")
    print(f"J ottima: {J_min:e}")
    print(f"Error Rate: {error_rate * 100:.1f}%")
    print(f"DCF min: {min_dcf}")
    print(f"Actual DCF:{act_dcf}\n")


Lambda: 0.0001
J ottima: 1.513656e-01
Error Rate: 6.0%
DCF min: 0.2602246543778802
Actual DCF:0.5887096774193548

Lambda: 0.00031622776601683794
J ottima: 1.534955e-01
Error Rate: 6.0%
DCF min: 0.2612167178699437
Actual DCF:0.5887096774193548

Lambda: 0.001
J ottima: 1.596177e-01
Error Rate: 5.9%
DCF min: 0.25865655401945725
Actual DCF:0.57864503328213

Lambda: 0.0031622776601683794
J ottima: 1.751964e-01
Error Rate: 5.9%
DCF min: 0.2527041730670763
Actual DCF:0.5614919354838709

Lambda: 0.01
J ottima: 2.086068e-01
Error Rate: 5.9%
DCF min: 0.2487359190988223
Actual DCF:0.5453309011776754

Lambda: 0.03162277660168379
J ottima: 2.686447e-01
Error Rate: 5.9%
DCF min: 0.243631592421915
Actual DCF:0.5049283154121864

Lambda: 0.1
J ottima: 3.594710e-01
Error Rate: 6.0%
DCF min: 0.24660778289810548
Actual DCF:0.5079045058883769

Lambda: 0.31622776601683794
J ottima: 4.716574e-01
Error Rate: 6.1%
DCF min: 0.2629128264208909
Actual DCF:0.5412186379928315

Lambda: 1.0
J ottima: 5.762647e-01
Err

Salviamo il miglior risultato ottenuto, ci servira nei notebook successivi

In [10]:
pi_T=0.1
target_prior_log_odds=np.log(pi_T/(1-pi_T))
l=0.03162277660168379
v_opt,J_min=trainLogReg(DTR_quad,LTR,l)
w_opt=v_opt[:-1]
b_opt=v_opt[-1]
S_val=np.dot(w_opt.T,DVAL_quad)+b_opt
LP=(S_val>0).astype(int)
error_rate=np.mean(LP!=LVAL)
score_lr_quad=S_val-target_prior_log_odds
np.save('llr/score_lr_best.npy', score_lr_quad)